In [7]:
!pip install -q gradio langchain-qdrant langchain-ollama sentence-transformers python-dotenv torch

In [8]:
import os
import shutil
from kaggle_secrets import UserSecretsClient

DATASET_PATH = "/kaggle/input/datasets/okharitonov/uesp-wiki-dataset" 

# 2. Копируем файлы скриптов в рабочую директорию (файл .env больше копировать не нужно!)
files_to_copy = ["rag_core.py", "app.py"]
for file in files_to_copy:
    src = os.path.join(DATASET_PATH, file)
    if os.path.exists(src):
        shutil.copy(src, "./")
        print(f"✓ Скопирован файл: {file}")
    else:
        print(f"⚠ Внимание: файл {file} не найден по пути {src}")

# 3. Загружаем секреты из Kaggle Secrets и прописываем их в системное окружение (os.environ)
user_secrets = UserSecretsClient()

expected_secrets = [
    "COLLECTION_NAME",
    "CROSS_ENCODER_MODEL_NAME",
    "EMBEDDING_MODEL_NAME",
    "OLLAMA_BASE_URL",
    "OLLAMA_MODEL",
    "QDRANT_API_KEY",
    "QDRANT_URL",
    "RERANK_TOP_N",
    "RETRIEVE_K"
]

print("\nЗагрузка секретов...")
for key in expected_secrets:
    try:
        # Пытаемся получить секрет из Kaggle
        val = user_secrets.get_secret(key)
        if val:
            os.environ[key] = str(val)
            # Маскируем API-ключ в логах для безопасности
            display_val = val[:10] + "..." if key == "QDRANT_API_KEY" else val
            print(f"  ✓ Секрет {key} загружен: '{display_val}'")
    except Exception:
        # Если секрет не задан в Kaggle, это нормально для необязательных параметров
        print(f"  ○ Секрет {key} не найден в Kaggle Secrets (будет использован дефолт из кода)")

# 4. Принудительно настраиваем параметры под T4 x2 прямо в окружении
os.environ["MODEL_DEVICE"] = "cuda:1"
print("\n✓ Устройство для тяжелых моделей принудительно выставлено на: cuda:1")

# 5. Переписываем app.py для генерации публичной ссылки в Kaggle (share=True)
if os.path.exists("app.py"):
    with open("app.py", "r") as f:
        app_code = f.read()

    app_code = app_code.replace("share=False", "share=True")

    with open("app.py", "w") as f:
        f.write(app_code)
    print("✓ Файл app.py успешно модифицирован для генерации публичной ссылки.")

print("\n=== Подготовка завершена! Все переменные успешно инжектированы в память ===")

✓ Скопирован файл: rag_core.py
✓ Скопирован файл: app.py

Загрузка секретов...
  ✓ Секрет COLLECTION_NAME загружен: 'oblivion_lore'
  ✓ Секрет CROSS_ENCODER_MODEL_NAME загружен: 'BAAI/bge-reranker-v2-m3'
  ✓ Секрет EMBEDDING_MODEL_NAME загружен: 'BAAI/bge-m3'
  ✓ Секрет OLLAMA_BASE_URL загружен: 'http://localhost:11434'
  ✓ Секрет OLLAMA_MODEL загружен: 'gemma3:12b'
  ✓ Секрет QDRANT_API_KEY загружен: 'eyJhbGciOi...'
  ✓ Секрет QDRANT_URL загружен: 'https://b50da76c-c016-433d-b4dd-93f354851313.europe-west3-0.gcp.cloud.qdrant.io'
  ✓ Секрет RERANK_TOP_N загружен: '5'
  ✓ Секрет RETRIEVE_K загружен: '20'

✓ Устройство для тяжелых моделей принудительно выставлено на: cuda:1
✓ Файл app.py успешно модифицирован для генерации публичной ссылки.

=== Подготовка завершена! Все переменные успешно инжектированы в память ===


In [ ]:
import subprocess
import time

# 1. Установка системных утилит
!sudo apt-get update && sudo apt-get install -y zstd pciutils

# 2. Установка Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Запуск Ollama на GPU 0
print("Запуск Ollama сервера на GPU 0...")
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_KEEP_ALIVE"] = "30m"
env["OLLAMA_NUM_PARALLEL"] = "4"

with open("ollama.log", "w") as log_file:
    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

time.sleep(5)

# 4. Скачивание модели
!ollama pull gemma3:12b

In [4]:
print("Запуск Ollama сервера на GPU 0...")
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_KEEP_ALIVE"] = "30m"
env["OLLAMA_NUM_PARALLEL"] = "4"

with open("ollama.log", "w") as log_file:
    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

Запуск Ollama сервера на GPU 0...


In [10]:
!curl http://localhost:11434

Ollama is running

In [11]:
!python app.py

[Retriever] Устройство: cuda:1
[Retriever] Загрузка эмбеддингов BAAI/bge-m3...
Loading weights: 100%|█| 391/391 [00:00<00:00, 2642.47it/s, Materializing param=
[Retriever] Загрузка реранкера BAAI/bge-reranker-v2-m3...
Loading weights: 100%|█| 393/393 [00:00<00:00, 737.87it/s, Materializing param=r
[Retriever] Подключение к Qdrant, коллекция 'oblivion_lore'...
[RAGPipeline] Подключение к Ollama (gemma3:12b @ http://localhost:11434)...
[RAGPipeline] Пайплайн полностью готов к работе.
/kaggle/working/app.py:107: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=480, type="messages", label="Диалог")
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://e03782b27eeaf1531f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio dep